# Fatia 1 — cadastro de clientes da XPTO

Explorar as camadas que o job `jobs/fatia1_clientes.py` gravou em `data/delta/`. Rode o job antes (ver README).

Rode sobre a amostra sintética (`seeds/amostra_sintetica/`); com dado real, não salve o notebook com outputs (constitution).

In [ ]:
from pyspark.sql import functions as F
from estudo.sessao import criar_sessao

spark = criar_sessao('notebook')
silver = spark.read.format('delta').load('/work/data/delta/silver_clientes')
silver.printSchema()

In [ ]:
# Como o documento chega do ERP e como fica
silver.groupBy('documento_tipo', 'documento_valido').count().orderBy(F.desc('count')).show()

In [ ]:
# Os números que mais se repetem entre clientes (telefone ou celular)
pares = (silver.select('codigo_cliente', F.col('telefone_fim8').alias('fim'))
         .unionByName(silver.select('codigo_cliente', F.col('celular_fim8').alias('fim')))
         .where('fim is not null').distinct())
pares.groupBy('fim').agg(F.countDistinct('codigo_cliente').alias('clientes')).orderBy(F.desc('clientes')).show(10)

In [ ]:
# A gold por dia, e o histórico do Delta (time travel)
gold = spark.read.format('delta').load('/work/data/delta/gold_reconhecimento')
gold.orderBy('dia_referencia', 'base').show()
spark.sql("DESCRIBE HISTORY delta.`/work/data/delta/silver_clientes`").select('version', 'timestamp', 'operation', 'operationMetrics').show(truncate=False)